# Cloud mask contingency frames, 801 vs 802 vs NWCSAF

Per hour: the NWCSAF cloud mask, each experiment's CLCT thresholded to a mask, and a
contingency map per experiment.

The threshold is the **two-year, per-retrieval-condition** `tau` fitted on the operational
KENDA-CH1 first guess in `CLCT_tau_conditions.ipynb` -- day 62 %, twilight 74.5 %,
night 74 %, applied per *pixel* by that pixel's own `ctth_conditions` flag. Not the
cross-validation fold values: those are fitted on half the data and are noisier, the
cross-validation only established that the procedure generalises.

Both experiments get the same `tau`. It describes the satellite retrieval, not the
experiment, so sharing it keeps the 801-802 difference free of any threshold artefact.

Sections 1-4 are copied verbatim from `CLCT_continuous_comparison.ipynb` -- same grid,
parallax correction, regridding and `TAU_COND`.


# Continuous CLCT comparison: NWCSAF cloud amount vs ICON

The companion notebook (`CLCT_threshold_corrected.ipynb`) goes one way: it binarises ICON's
continuous CLCT at a fitted threshold so it can be scored against the binary NWCSAF mask.
This notebook goes the other way, and uses **no threshold at all**.

`cma` is binary per 3 km pixel, but *averaged over many pixels* it is a continuous cloud
fraction — exactly the quantity CLCT already is. So:

1. **Cloud amount over Switzerland.** Average `cma` over all Swiss cells -> one number per hour,
   continuous, no tuned parameter. Compare with ICON's CLCT averaged over the same cells.
   This is the headline timeseries, and it is directly comparable to the MOVERO octa bias.
2. **3x coarsening.** For a *map* the average has to be local, so both fields are block-averaged
   over 3x3 cells (~9 km). Both fields, identically — coarsening only one would compare a smoothed
   observation against a point forecast and make ICON look spuriously too extreme.

**The one caveat to keep in mind:** a pixel that is 20% cloudy still reads `cma = 1`. So the
satellite cloud amount is structurally *higher* than a true areal fraction, most in broken cloud.
That makes the absolute bias hard to read — but it is a near-constant offset, so it cancels in the
two things this project cares about: the *variation with lead time* and the *801 - 802 difference*.
It is also the same effect that pushed `tau*` to 66/77% rather than 50%.

Everything upstream (conservative regridding, T-15 slot matching, parallax, snow/ice screening)
is unchanged from the threshold notebook and justified there / in `docs/conventions.md`.

In [ ]:
import os
os.environ["ECCODES_VERSION_CHECK_OFF"] = "1"

from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import xarray as xr
import shapely
import earthkit.data as ekd

import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.io import shapereader

if shapely.__version__ < "2":
    raise RuntimeError(f"needs shapely >= 2, found {shapely.__version__}")

# =============================================================================
# Config
# =============================================================================
EXPS = ["801", "802"]
NWCSAF_DIR = "/scratch/mch/jdelbeke/nwcsaf"
ICON_DIR_T = "/store_new/mch/msopr/jdelbeke/ICON_TST/{exp}/FG25/det"
ICON_GRID_FILE = "/oprusers/osm/opr.inn/data/ICON_INPUT/ICON-CH1-EPS/icon_grid_0001_R19B08_mch.nc"

CACHE_DIR = "/scratch/mch/jdelbeke"
WEIGHTS_CACHE = f"{CACHE_DIR}/regrid_weights_icon_nwcsaf.npz"   # shared with the threshold notebook
SERIES_CACHE = f"{CACHE_DIR}/clct_continuous_series.npz"


def nwcsaf_file(t):
    return f"{NWCSAF_DIR}/MSG_nwcsaf_cosmo1eqc3km_{t:%Y%m%d%H%M}.nc"


def icon_file(exp, t):
    return f"{ICON_DIR_T.format(exp=exp)}/lff{t:%Y%m%d%H}"


START = datetime(2025, 10, 4, 1)
N_HOURS = 144
TIMES = [START + timedelta(hours=h) for h in range(N_HOURS)]

DOMAIN = None            # None = whole NWCSAF grid
MIN_COVERAGE = 0.999     # target cells less covered by ICON than this -> NaN
CHUNK = 200_000

# --- corrections (identical to the threshold notebook) -----------------------
PARALLAX = True
NWCSAF_SLOT_LAG = timedelta(minutes=15)   # verified; see docs/conventions.md
CT_SNOW_ICE = (3, 4)                      # Snow_over_land, Sea_ice -- the only screen
SAT_LON = 0.0

# --- coarsening --------------------------------------------------------------
COARSEN = 5              # 5x5 block average, ~15 km, applied to BOTH fields
# 3x was tried first and rejected: a 3x3 block of binary pixels takes only 10
# distinct values, and cloud is spatially correlated, so most blocks came out
# all-cloud or all-clear and the observation still read as a hard mask beside
# ICON's smooth field. 5x5 gives 26 levels. This changes no number in the
# headline timeseries (computed at native 3 km) -- only the maps and the
# coarse RMSE/correlation columns.
MIN_BLOCK_FRAC = 0.5     # a coarse cell needs this fraction of its fine cells valid

OCTA = 12.5              # 1 okta, for the secondary axis against MOVERO

print(f"{len(TIMES)} hourly steps, {TIMES[0]} .. {TIMES[-1]}; experiments {EXPS}")

## 1. Regridding, geometry and the NWCSAF reader

Unchanged from `CLCT_threshold_corrected.ipynb`; the weight cache is shared.

In [ ]:
def read_grid(path, domain=None):
    with xr.open_dataset(path) as ds:
        lon2 = np.asarray(ds["longitude"].values)
        lat2 = np.asarray(ds["latitude"].values)
    if lon2.ndim == 2:
        if (np.abs(lon2 - lon2[0, :]).max() > 1e-9
                or np.abs(lat2 - lat2[:, [0]]).max() > 1e-9):
            raise ValueError("2-D coordinates are not a regular lat/lon grid")
        lon, lat = lon2[0, :], lat2[:, 0]
    else:
        lon, lat = lon2, lat2
    dlon = float(np.median(np.diff(lon)))
    dlat = float(np.median(np.diff(lat)))     # negative: latitude descends
    if (np.abs(np.diff(lon) - dlon).max() > 1e-9
            or np.abs(np.diff(lat) - dlat).max() > 1e-9):
        raise ValueError("grid spacing is not uniform")
    if domain is not None:
        jx = np.where((lon >= domain[0]) & (lon <= domain[1]))[0]
        iy = np.where((lat >= domain[2]) & (lat <= domain[3]))[0]
        lon, lat = lon[jx], lat[iy]
    return lon, lat, dlon, dlat


def read_icon_cells(grid_file, n_expected, lon_c, lat_c):
    with xr.open_dataset(grid_file) as g:
        vlon = np.asarray(g["clon_vertices"].values, dtype="float64")
        vlat = np.asarray(g["clat_vertices"].values, dtype="float64")
    if vlon.shape[0] != n_expected:
        raise ValueError(f"grid file has {vlon.shape[0]} cells, field has {n_expected}")
    if np.abs(vlat).max() <= np.pi + 1e-6:
        vlon, vlat = np.rad2deg(vlon), np.rad2deg(vlat)
    dv = max(np.abs(vlon.mean(axis=1) - lon_c).max(),
             np.abs(vlat.mean(axis=1) - lat_c).max())
    if dv > 1e-3:
        raise ValueError("vertices do not match the field's cell centres")
    return vlon, vlat


def triangle_polygons(vlon, vlat, sub):
    rings = np.stack([vlon[sub], vlat[sub]], axis=-1)
    rings = np.concatenate([rings, rings[:, :1, :]], axis=1)
    return shapely.polygons(rings)


def build_weights(lon, lat, dlon, dlat, vlon, vlat):
    ny, nx = lat.size, lon.size
    ncell = ny * nx
    hx, hy = abs(dlon) / 2, abs(dlat) / 2
    near = ((vlon.max(axis=1) >= lon.min() - hx) & (vlon.min(axis=1) <= lon.max() + hx)
            & (vlat.max(axis=1) >= lat.min() - hy) & (vlat.min(axis=1) <= lat.max() + hy))
    sub = np.where(near)[0]
    tri = triangle_polygons(vlon, vlat, sub)
    LON, LAT = np.meshgrid(lon, lat)
    cells = shapely.box((LON - hx).ravel(), (LAT - hy).ravel(),
                        (LON + hx).ravel(), (LAT + hy).ravel())
    ti, ci = shapely.STRtree(cells).query(tri, predicate="intersects")
    w = np.empty(ti.size)
    for a in range(0, ti.size, CHUNK):
        b = min(a + CHUNK, ti.size)
        w[a:b] = shapely.area(shapely.intersection(tri[ti[a:b]], cells[ci[a:b]]))
    keep = w > 0
    ti, ci, w = ti[keep], ci[keep], w[keep]
    coverage = np.bincount(ci, weights=w, minlength=ncell) / (abs(dlon) * abs(dlat))
    return sub, ti, ci, w, coverage.reshape(ny, nx)


def remap(values, sub, ti, ci, w, coverage, shape):
    v = np.asarray(values, dtype="float64").ravel()[sub]
    ok = np.isfinite(v[ti])
    ncell = shape[0] * shape[1]
    num = np.bincount(ci[ok], weights=w[ok] * v[ti][ok], minlength=ncell)
    den = np.bincount(ci[ok], weights=w[ok], minlength=ncell)
    out = np.full(ncell, np.nan)
    good = (coverage.ravel() >= MIN_COVERAGE) & (den > 0)
    out[good] = num[good] / den[good]
    return out.reshape(shape)

In [ ]:
# --- grid, weights, orography -------------------------------------------------
lon, lat, dlon, dlat = read_grid(nwcsaf_file(TIMES[0]), DOMAIN)
shape = (lat.size, lon.size)

fl = ekd.from_source("file", icon_file(EXPS[0], TIMES[0])).to_fieldlist()
xa = fl.sel({"parameter.variable": "CLCT"})[0].to_xarray()
name0 = "CLCT" if "CLCT" in xa else list(xa.data_vars)[0]
lon_c = np.asarray(xa["longitude"].values).ravel()
lat_c = np.asarray(xa["latitude"].values).ravel()
vlon, vlat = read_icon_cells(ICON_GRID_FILE, lon_c.size, lon_c, lat_c)

if WEIGHTS_CACHE and os.path.exists(WEIGHTS_CACHE):
    z = np.load(WEIGHTS_CACHE)
    sub, ti, ci, w, coverage = z["sub"], z["ti"], z["ci"], z["w"], z["coverage"]
    print(f"weights loaded from {WEIGHTS_CACHE}")
else:
    sub, ti, ci, w, coverage = build_weights(lon, lat, dlon, dlat, vlon, vlat)
    if WEIGHTS_CACHE:
        np.savez_compressed(WEIGHTS_CACHE, sub=sub, ti=ti, ci=ci, w=w, coverage=coverage)
in_domain = coverage >= MIN_COVERAGE

xh = fl.sel({"parameter.variable": "HSURF"})[0].to_xarray()
nmh = "HSURF" if "HSURF" in xh else list(xh.data_vars)[0]
HSURF = remap(np.asarray(xh[nmh].values, dtype="float64").ravel(),
              sub, ti, ci, w, coverage, shape)
HSURF_KM = np.where(np.isfinite(HSURF), HSURF, 0.0) / 1000.0

print(f"target grid {shape}, {int(in_domain.sum())} cells inside the ICON domain")

In [ ]:
# --- parallax geometry and scan timing ---------------------------------------
R_E, R_S = 6371.0, 42164.0
Y_DISC = 0.1518

LON2, LAT2 = np.meshgrid(lon, lat)
_la, _dlo = np.deg2rad(LAT2), np.deg2rad(LON2 - SAT_LON)
PSI = np.arccos(np.cos(_la) * np.cos(_dlo))
VZA = np.arctan2(R_S * np.sin(PSI), R_S * np.cos(PSI) - R_E)
BRG = np.arctan2(np.sin(-_dlo), -np.sin(_la) * np.cos(-_dlo))
DLAT_PER_KM = np.tan(VZA) * np.cos(BRG) / 111.32
DLON_PER_KM = np.tan(VZA) * np.sin(BRG) / (111.32 * np.cos(_la))


def geos_y_angle(lat_deg, lon_deg, sat_lon=SAT_LON):
    c_lat = np.arctan(0.993243 * np.tan(np.deg2rad(lat_deg)))
    r_l = 6356.5838 / np.sqrt(1 - 0.00669438 * np.cos(c_lat) ** 2)
    dl = np.deg2rad(lon_deg - sat_lon)
    r1 = 42164.0 - r_l * np.cos(c_lat) * np.cos(dl)
    r2 = -r_l * np.cos(c_lat) * np.sin(dl)
    r3 = r_l * np.sin(c_lat)
    return np.arcsin(-r3 / np.sqrt(r1 ** 2 + r2 ** 2 + r3 ** 2))


SCAN_FRAC = float((Y_DISC - geos_y_angle(float(np.mean(lat)), float(np.mean(lon))))
                  / (2 * Y_DISC))


def match_slot(t):
    """Strict T-15 pairing; an hour with no such slot is dropped, not re-paired."""
    slot = t - NWCSAF_SLOT_LAG
    return slot if os.path.exists(nwcsaf_file(slot)) else None


_c = (shape[0] // 2, shape[1] // 2)
print(f"viewing zenith at domain centre {np.rad2deg(VZA[_c]):.1f} deg -> "
      f"displacement {np.tan(VZA[_c]):.2f} x height")
print(f"domain imaged {100 * SCAN_FRAC:.0f}% through the sweep; pairing slot "
      f"T-{int(NWCSAF_SLOT_LAG.total_seconds() // 60)}min with model hour T")

In [ ]:
def decode_flags(da):
    a = np.asarray(da.values)
    ai = np.where(np.isfinite(a), a, 0).astype("int64")
    meanings = da.attrs["flag_meanings"].split()
    values = np.asarray(da.attrs["flag_values"]).astype("int64")
    masks = np.asarray(da.attrs.get("flag_mask", da.attrs.get("flag_masks", values))).astype("int64")
    out, seen = {}, {}
    for m, v, mk in zip(meanings, values, masks):
        seen[m] = seen.get(m, 0) + 1
        out[m if seen[m] == 1 else f"{m}__{seen[m]}"] = (ai & int(mk)) == int(v)
    return out


def read_nwcsaf(path, domain=None):
    with xr.open_dataset(path) as ds:
        sl = (slice(None), slice(None))
        cma = np.asarray(ds["cma"].squeeze().values, dtype="float64")[sl]
        cma[cma == 255] = np.nan
        ct = np.asarray(ds["ct"].squeeze().values, dtype="float64")[sl]
        alti = np.asarray(ds["ctth_alti"].squeeze().values, dtype="float64")[sl]
        lo, hi = ds["ctth_alti"].attrs.get("valid_range", (-2000.0, 25000.0))
        alti[(alti < lo) | (alti > hi)] = np.nan
        meth = {k: v[sl] for k, v in decode_flags(ds["ctth_method"]).items()}
        cond = {k: v[sl] for k, v in decode_flags(ds["ctth_conditions"]).items()}
    return dict(cma=cma, ct=ct, alti=alti, meth=meth, cond=cond)


def parallax_correct(sat, hsurf_km):
    """Displace the mask onto true ground positions. Cells that only an
    uncorrectable cloudy pixel could occupy become NaN (unobserved), not clear."""
    cma = sat["cma"]
    ny, nx = cma.shape
    cloudy, clear = cma == 1, cma == 0
    h_km = sat["alti"] / 1000.0
    ok_h = np.isfinite(h_km) & ~sat["meth"].get("No_reliable_method", np.zeros(cma.shape, bool))
    movable, lost = cloudy & ok_h, cloudy & ~ok_h

    def scatter(mask, z_km):
        i0, j0 = np.nonzero(mask)
        lat_t = LAT2[i0, j0] + DLAT_PER_KM[i0, j0] * z_km[i0, j0]
        lon_t = LON2[i0, j0] + DLON_PER_KM[i0, j0] * z_km[i0, j0]
        i1 = np.rint((lat_t - lat[0]) / dlat).astype(int)
        j1 = np.rint((lon_t - lon[0]) / dlon).astype(int)
        keep = (i1 >= 0) & (i1 < ny) & (j1 >= 0) & (j1 < nx)
        cnt = np.zeros((ny, nx), dtype=np.int32)
        np.add.at(cnt, (i1[keep], j1[keep]), 1)
        return cnt

    n_cloud = scatter(movable, np.where(ok_h, h_km, 0.0))
    n_clear = scatter(clear, hsurf_km)
    n_lost = scatter(lost, hsurf_km)
    out = np.full((ny, nx), np.nan)
    out[n_clear > 0] = 0.0
    out[n_cloud > 0] = 1.0                       # cloud wins a collision
    out[(n_lost > 0) & (n_cloud == 0)] = np.nan
    return out


def observed_fraction(slot):
    """NWCSAF cloud amount for one slot, in percent: 0/100 per cell, NaN where
    unobserved. Screened for snow/ice only."""
    S = read_nwcsaf(nwcsaf_file(slot), DOMAIN)
    cma = parallax_correct(S, HSURF_KM) if PARALLAX else S["cma"]
    cma = np.where(np.isin(S["ct"], CT_SNOW_ICE), np.nan, cma)
    return 100.0 * cma, S

## 2. Switzerland mask and the 3x coarsening

The timeseries is a Swiss average, to sit alongside MOVERO's `ch` subset. Both fields are coarsened with the *same* operator and the *same* validity mask, so the comparison stays scale-symmetric.

In [ ]:
_fn = shapereader.natural_earth(resolution="10m", category="cultural",
                                name="admin_0_countries")
_CH_GEOM = [r.geometry for r in shapereader.Reader(_fn).records()
            if r.attributes.get("ADMIN") == "Switzerland"][0]
CH_MASK = shapely.contains_xy(_CH_GEOM, LON2, LAT2) & in_domain


def block_mean(a, k=COARSEN, min_frac=MIN_BLOCK_FRAC):
    """Non-overlapping k x k block average, NaN where too few fine cells are valid."""
    ny, nx = a.shape
    ny2, nx2 = (ny // k) * k, (nx // k) * k
    b = np.asarray(a, dtype="float64")[:ny2, :nx2].reshape(ny2 // k, k, nx2 // k, k)
    v = np.isfinite(b)
    n = v.sum(axis=(1, 3))
    s = np.where(v, b, 0.0).sum(axis=(1, 3))
    out = np.full(n.shape, np.nan)
    ok = n >= max(1, int(np.ceil(min_frac * k * k)))
    out[ok] = s[ok] / n[ok]
    return out


def block_coord(c, k=COARSEN):
    n = (c.size // k) * k
    return c[:n].reshape(-1, k).mean(axis=1)


lon_c3, lat_c3 = block_coord(lon), block_coord(lat)
CH_MASK_C = block_mean(CH_MASK.astype(float)) >= 0.5
shape_c = (lat_c3.size, lon_c3.size)

print(f"Switzerland: {int(CH_MASK.sum())} cells at {abs(dlon) * 111.32 * np.cos(np.deg2rad(46.8)):.1f} km, "
      f"{int(CH_MASK_C.sum())} cells at {COARSEN}x ({COARSEN * abs(dlon) * 111.32 * np.cos(np.deg2rad(46.8)):.1f} km)")
print(f"a coarse cell can take {COARSEN ** 2 + 1} distinct observed values "
      f"(0, 1/{COARSEN ** 2}, ... 1) -- the quantisation floor of this method")

## 3. One pass over the period

Per hour: the Swiss-mean cloud amount from the satellite and from each experiment, over the **same cells** (a cell the satellite could not see is excluded from both). Plus the coarsened pointwise errors.

In [ ]:
# NWCSAF applies a DIFFERENT sequence of spectral tests in each illumination
# regime (day: solar elevation > 10 deg, twilight: -3..10, night: < -3; see the
# CMA product description). So a diurnal curve of NWCSAF-derived bias is a curve
# through three different instruments, and a step at a regime boundary is an
# instrument change, not model behaviour. Every mean below is therefore also
# accumulated separately over the pixels of each regime, using the file's own
# ctth_conditions flags, so that like is only ever compared with like.
REGIMES = ["day", "twilight", "night"]
MIN_REGIME_CELLS = 200      # Swiss cells needed before an hour counts for a regime

# Optimal CLCT thresholds by retrieval condition, fitted on two years of
# OPERATIONAL KENDA-CH1 det FG against NWCSAF (CLCT_tau_conditions.ipynb,
# 16795 hourly scenes, Sep 2024 - Sep 2026). Held-out J showed that adding a
# calendar-month dimension buys only +0.004 J, so these three numbers are the
# whole threshold model. The same taus are applied to 801 and 802: they
# describe the satellite retrieval, not the experiment, so using one set keeps
# the 801-802 difference free of any threshold artefact.
TAU_COND = {"day": 62.0, "twilight": 74.5, "night": 74.0}
TAU_FALLBACK = 68.0         # pixels NWCSAF assigns to none of the three

# Trust flag (section 5c). Pure instrument physics, nothing fitted. The two
# references fail on OPPOSITE cloud types, so the flag says which to prefer:
#   thin ice cloud   -> SEVIRI sees cold ice against a warm background easily,
#                       while the MOVERO reference (APCADA, from longwave
#                       downward radiation) gets almost no signal from optically
#                       thin cirrus. Prefer the satellite.
#   night low cloud  -> a warm-topped deck is a strong APCADA signal but kills
#                       the IR10.8-IR3.9 contrast SEVIRI needs. Prefer the
#                       stations.
CIRRUS_FRAC, STRATUS_CLD, STRATUS_TOP = 0.35, 0.45, 3500.0
CT_SEMITRANSP = (10, 11, 12, 13, 14)   # NWCSAF CT semi-transparent classes

SIG = "|".join([",".join(EXPS), str(START), str(N_HOURS), str(MIN_COVERAGE),
                str(PARALLAX), str(NWCSAF_SLOT_LAG), str(COARSEN),
                str(MIN_BLOCK_FRAC), str(CT_SNOW_ICE), ",".join(REGIMES),
                str(MIN_REGIME_CELLS), str(sorted(TAU_COND.items())),
                str(TAU_FALLBACK), str((CIRRUS_FRAC, STRATUS_CLD, STRATUS_TOP)),
                str(CT_SEMITRANSP)])


def load_clct(exp, t):
    fl_t = ekd.from_source("file", icon_file(exp, t)).to_fieldlist()
    sel = fl_t.sel({"parameter.variable": "CLCT"})
    if len(sel) == 0:
        raise FileNotFoundError("no CLCT field")
    xa_t = sel[0].to_xarray()
    nm = "CLCT" if "CLCT" in xa_t else list(xa_t.data_vars)[0]
    return remap(np.asarray(xa_t[nm].values, dtype="float64").ravel(),
                 sub, ti, ci, w, coverage, shape)


def hour_stats(t):
    """Swiss-mean cloud amount and coarsened errors for one model hour."""
    slot = match_slot(t)
    if slot is None:
        raise FileNotFoundError("no T-15 slot")
    obs, S = observed_fraction(slot)
    clct = {e: load_clct(e, t) for e in EXPS}

    # one common sample: Swiss cells the satellite saw and every experiment covers
    common = CH_MASK & np.isfinite(obs)
    for e in EXPS:
        common &= np.isfinite(clct[e])

    row = {"time": t, "slot": slot, "n_cells": int(common.sum()),
           "unobserved": 1 - common.sum() / max(int(CH_MASK.sum()), 1),
           "obs_mean": float(obs[common].mean())}

    # --- the same means, but over the pixels of one illumination regime only ---
    for rg in REGIMES:
        rm = common & S["cond"].get(rg, np.zeros(shape, bool))
        n_rg = int(rm.sum())
        row[f"n_{rg}"] = n_rg
        ok_rg = n_rg >= MIN_REGIME_CELLS
        row[f"obs_{rg}"] = float(obs[rm].mean()) if ok_rg else np.nan
        for e in EXPS:
            row[f"{e}_{rg}"] = float(clct[e][rm].mean()) if ok_rg else np.nan

    # --- satellite descriptors behind the trust flag ------------------------
    cloudy = common & (obs >= 50.0)
    _a = S["alti"][cloudy & np.isfinite(S["alti"])]
    row["alti_mean"] = float(_a.mean()) if _a.size > 50 else np.nan
    row["ct_semitransp"] = (float(np.isin(S["ct"][cloudy], CT_SEMITRANSP).mean())
                            if int(cloudy.sum()) > 50 else np.nan)
    row["f_night_px"] = float(
        (common & S["cond"].get("night", np.zeros(shape, bool))).sum()
        / max(int(common.sum()), 1))

    # --- binary bias: ICON thresholded at its own pixel's condition tau -----
    # Both fields are 0/100 here, so the difference of the means is a
    # cloud-amount bias in the same units as the continuous row above.
    tau_px = np.full(shape, TAU_FALLBACK)
    for rg in REGIMES:
        tau_px[S["cond"].get(rg, np.zeros(shape, bool))] = TAU_COND[rg]
    for e in EXPS:
        row[f"{e}_thr_mean"] = float(
            100.0 * (clct[e][common] >= tau_px[common]).mean())

    # coarsen only the common sample, so both fields see identical blocks
    obs_c = block_mean(np.where(common, obs, np.nan))
    ok_c = CH_MASK_C & np.isfinite(obs_c)
    row["obs_mean_c"] = float(obs_c[ok_c].mean()) if ok_c.any() else np.nan

    for e in EXPS:
        m = clct[e]
        row[f"{e}_mean"] = float(m[common].mean())
        d = m[common] - obs[common]
        row[f"{e}_rmse"] = float(np.sqrt((d ** 2).mean()))
        m_c = block_mean(np.where(common, m, np.nan))
        oc = ok_c & np.isfinite(m_c)
        if oc.sum() > 2:
            dc = m_c[oc] - obs_c[oc]
            row[f"{e}_rmse_c"] = float(np.sqrt((dc ** 2).mean()))
            row[f"{e}_corr_c"] = float(np.corrcoef(m_c[oc], obs_c[oc])[0, 1])
        else:
            row[f"{e}_rmse_c"] = row[f"{e}_corr_c"] = np.nan
    return row


rows = None
if SERIES_CACHE and os.path.exists(SERIES_CACHE):
    _z = np.load(SERIES_CACHE, allow_pickle=True)
    if str(_z["signature"]) == SIG:
        rows = list(_z["rows"])
        print(f"series loaded from {SERIES_CACHE} ({len(rows)} hours)")
    else:
        print("cache built with different settings -- rebuilding")

if rows is None:
    rows = []
    for i, t in enumerate(TIMES, 1):
        try:
            rows.append(hour_stats(t))
        except Exception as e:
            print(f"  skip {t:%m-%d %Hh}: {type(e).__name__}: {e}")
        if i % 24 == 0:
            print(f"  {i}/{len(TIMES)}", flush=True)
    if SERIES_CACHE:
        np.savez_compressed(SERIES_CACHE, rows=np.array(rows, dtype=object),
                            signature=np.array(SIG))

SER = pd.DataFrame(rows).set_index("time").sort_index()
for e in EXPS:
    SER[f"{e}_bias"] = SER[f"{e}_mean"] - SER["obs_mean"]
    SER[f"{e}_bias_thr"] = SER[f"{e}_thr_mean"] - SER["obs_mean"]
    for rg in REGIMES:
        SER[f"{e}_bias_{rg}"] = SER[f"{e}_{rg}"] - SER[f"obs_{rg}"]
SER["hour"] = SER.index.hour


def trust_flag(r):
    """Which observation is expected to fail this hour."""
    if r["ct_semitransp"] >= CIRRUS_FRAC:
        return "CIRRUS"
    if (r["f_night_px"] > 0.5 and r["obs_mean"] < 100 * STRATUS_CLD
            and r["alti_mean"] < STRATUS_TOP):
        return "STRATUS"
    return "AGREE"


SER["flag"] = SER.apply(trust_flag, axis=1)
print("\ntrust flag: "
      + ",  ".join(f"{k} {v}h" for k, v in SER["flag"].value_counts().items()))

print(f"\n{len(SER)} hours, {SER.index[0]} .. {SER.index[-1]}")
print(f"Swiss cells used per hour: {SER['n_cells'].min()}-{SER['n_cells'].max()} "
      f"of {int(CH_MASK.sum())}  (mean {SER['unobserved'].mean():.1%} unobserved)")
print(f"\nNWCSAF cloud amount over Switzerland: mean {SER['obs_mean'].mean():.1f}%")
for e in EXPS:
    print(f"  {e}: CLCT {SER[f'{e}_mean'].mean():5.1f}%   bias "
          f"{SER[f'{e}_bias'].mean():+5.1f}% ({SER[f'{e}_bias'].mean() / OCTA:+.2f} octa)"
          f"   RMSE {SER[f'{e}_rmse'].mean():.1f}% (3km) -> "
          f"{SER[f'{e}_rmse_c'].mean():.1f}% ({COARSEN}x)"
          f"   corr {SER[f'{e}_corr_c'].mean():.2f}")

print(f"\nthresholded at tau by condition {TAU_COND} "
      f"(fallback {TAU_FALLBACK}%):")
for e in EXPS:
    print(f"  {e}: cloudy fraction {SER[f'{e}_thr_mean'].mean():5.1f}%   bias "
          f"{SER[f'{e}_bias_thr'].mean():+5.1f}% "
          f"({SER[f'{e}_bias_thr'].mean() / OCTA:+.2f} octa)")

In [ ]:
MOVERO_WD = Path("/scratch/mch/jdelbeke/movero/wd/2025_kenda_801_802")
# Repo standard (Okabe-Ito), matching the other notebooks. Do not substitute.
EXP_COLOUR = {"801": "#0072B2", "802": "#D55E00"}
VERIF_SUBSET = "ch"
VERIF_PARAM, VERIF_SCORE = "CLCT", "ME"


def read_movero_time_scores(exp_dir, param=VERIF_PARAM):
    path = Path(exp_dir) / f"time_scores01_{param}.dat"
    with open(path) as fh:
        lines = fh.readlines()
    i_hdr = next(i for i, line in enumerate(lines) if line.split()[:1] == ["YYYY"])
    names = lines[i_hdr].split()
    df = pd.read_csv(path, sep=r"\s+", skiprows=i_hdr + 1, names=names, header=None)
    df = df.mask(df < -1e8)
    df["time"] = pd.to_datetime(dict(year=df["YYYY"], month=df["MM"], day=df["DD"],
                                     hour=df["hh"], minute=df["mm"]))
    return df.set_index("time").sort_index()


bias = {e: read_movero_time_scores(MOVERO_WD / f"{e}-FG-det_{VERIF_SUBSET}")[VERIF_SCORE]
        for e in EXPS}

# MOBS is the station-mean observed cloud cover in octa -- the observation
# itself, identical for both experiments (same stations, same hours), so one
# series is enough. It answers a different question from the bias rows: is the
# model wrong in a way that depends on how cloudy it actually was?
obs_octa = read_movero_time_scores(
    MOVERO_WD / f"{EXPS[0]}-FG-det_{VERIF_SUBSET}")["MOBS"]

# Frozen limits, so the marker sits identically in every frame.
TSPAN = (SER.index[0], SER.index[-1])
_a = np.concatenate([b.dropna().values for b in bias.values()])
MOVERO_YLIM = (np.nanmin(_a) - 0.12 * np.ptp(_a), np.nanmax(_a) + 0.12 * np.ptp(_a))
_b = np.concatenate([SER[f"{e}_bias"].values for e in EXPS])
SAT_YLIM = (np.nanmin(_b) - 0.12 * np.ptp(_b), np.nanmax(_b) + 0.12 * np.ptp(_b))


def draw_sat_bias(ax, t=None, legend=True):
    """ICON CLCT minus NWCSAF cloud amount, Swiss mean, no threshold anywhere."""
    for e in EXPS:
        ax.plot(SER.index, SER[f"{e}_bias"], "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  CLCT - NWCSAF")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*TSPAN)
    ax.set_ylim(*SAT_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel("cloud-amount bias [%]")
    sec = ax.secondary_yaxis("right", functions=(lambda v: v / OCTA, lambda v: v * OCTA))
    sec.set_ylabel("[octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXPS:
        v = SER[f"{e}_bias"].get(ts, np.nan)
        if np.isfinite(v):
            ax.plot([ts], [v], marker="*", ms=20, color=EXP_COLOUR[e], mec="k",
                    mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {v:+.1f}%")
    ax.set_title("satellite: Swiss-mean CLCT - NWCSAF cloud amount; "
                 + ("now: " + ",  ".join(shown) if shown else "-"),
                 fontsize=10, loc="left")


def draw_movero_bias(ax, t=None, legend=True):
    for e in EXPS:
        ax.plot(bias[e].index, bias[e].values, "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  {VERIF_PARAM} {VERIF_SCORE}")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*TSPAN)
    ax.set_ylim(*MOVERO_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel(f"{VERIF_PARAM} bias [octa]")
    if legend:
        ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXPS:
        v = bias[e].get(ts, np.nan)
        if np.isfinite(v):
            ax.plot([ts], [v], marker="*", ms=20, color=EXP_COLOUR[e], mec="k",
                    mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {v:+.2f}")
    ax.set_title(f"MOVERO first-guess {VERIF_PARAM} bias, all Swiss stations; "
                 + ("now: " + ",  ".join(shown) + " octa" if shown
                    else "no verification at this hour"),
                 fontsize=10, loc="left")


for e in EXPS:
    print(f"{e}: MOVERO {VERIF_SCORE} mean {bias[e].mean():+.3f} octa; "
          f"satellite bias mean {SER[f'{e}_bias'].mean() / OCTA:+.3f} octa")

## 5. Frames

Four outcomes per pixel, from the model mask against the NWCSAF mask:

| | satellite cloudy | satellite clear |
|---|---|---|
| **model cloudy** | hit | false alarm |
| **model clear** | miss | correct negative |

Correct negatives are drawn at low alpha: they are usually most of the domain and would
otherwise swamp the three outcomes that carry information.


In [ ]:
import sys, subprocess
from concurrent.futures import ProcessPoolExecutor, as_completed
from matplotlib.figure import Figure
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

FRAME_DIR = Path("./figures/clct_contingency_frames")
FRAME_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DPI, N_WORKERS = 110, 16

# Masks: one ink for cloud, white for clear, the repo's amber for "not observed"
# (parallax gap or snow/ice screen) so clear and unobserved never share a colour.
MASK_CMAP = ListedColormap(["#ffffff", "#5b7fa6"])
MASK_CMAP.set_bad(NODATA_COLOUR := "#c98a3a")

# Contingency. Correct negatives are deliberately faint -- they are the bulk of
# the domain and carry the least information.
CAT_COLOURS = [(0.80, 0.80, 0.80, 0.25),   # 0 correct negative, translucent
               (0.17, 0.35, 0.52, 1.00),   # 1 hit
               (0.65, 0.06, 0.08, 1.00),   # 2 miss        (model too clear)
               (0.80, 0.47, 0.65, 1.00)]   # 3 false alarm (model too cloudy)
CAT_CMAP = ListedColormap(CAT_COLOURS)
CAT_CMAP.set_bad(NODATA_COLOUR)
CAT_NORM = BoundaryNorm([-0.5, 0.5, 1.5, 2.5, 3.5], CAT_CMAP.N)
CAT_LABEL = ["correct negative (both clear)", "hit (both cloudy)",
             "miss (satellite cloudy, model clear)",
             "false alarm (model cloudy, satellite clear)"]

OUTSIDE_CMAP = ListedColormap(["white"])
_ii, _jj = np.where(in_domain)
DOMAIN_EXTENT = [lon[_jj.min()] - 2 * abs(dlon), lon[_jj.max()] + 2 * abs(dlon),
                 lat[_ii.max()] - 2 * abs(dlat), lat[_ii.min()] + 2 * abs(dlat)]


def mask_outside(ax):
    ax.pcolormesh(LON2, LAT2, np.where(~in_domain, 1.0, np.nan),
                  transform=ccrs.PlateCarree(), cmap=OUTSIDE_CMAP,
                  shading="nearest", zorder=2)


def tau_field(S):
    """The per-condition threshold, per pixel, from the file's own flags."""
    t = np.full(shape, TAU_FALLBACK)
    for rg in REGIMES:
        t[S["cond"].get(rg, np.zeros(shape, bool))] = TAU_COND[rg]
    return t


def contingency(model_cloudy, obs_cloudy):
    """0 correct negative, 1 hit, 2 miss, 3 false alarm."""
    return np.where(obs_cloudy,
                    np.where(model_cloudy, 1, 2),
                    np.where(model_cloudy, 3, 0)).astype("float64")


# The satellite row shows the THRESHOLDED bias, so the timeseries measures the
# same masks the maps above it are built from.
_c = np.concatenate([SER[f"{e}_bias_thr"].dropna().values for e in EXPS])
THR_YLIM = (np.nanmin(_c) - 0.12 * np.ptp(_c), np.nanmax(_c) + 0.12 * np.ptp(_c))


def draw_thr_bias(ax, t=None):
    """ICON thresholded at the per-condition tau, minus the NWCSAF mask."""
    for e in EXPS:
        ax.plot(SER.index, SER[f"{e}_bias_thr"], "-", color=EXP_COLOUR[e], lw=1.3,
                label=f"{e}  CLCT>=tau - NWCSAF")
    ax.axhline(0, color="k", lw=0.9)
    ax.set_xlim(*TSPAN)
    ax.set_ylim(*THR_YLIM)
    ax.grid(alpha=0.3)
    ax.set_ylabel("cloud-amount bias [%]")
    sec = ax.secondary_yaxis("right", functions=(lambda v: v / OCTA, lambda v: v * OCTA))
    sec.set_ylabel("[octa]")
    ax.legend(fontsize=8, ncol=len(EXPS), loc="upper right")
    if t is None:
        return
    ts = pd.Timestamp(t)
    ax.axvline(ts, color="k", lw=1.4, ls=":", zorder=4)
    shown = []
    for e in EXPS:
        v = SER[f"{e}_bias_thr"].get(ts, np.nan)
        if np.isfinite(v):
            ax.plot([ts], [v], marker="*", ms=20, color=EXP_COLOUR[e], mec="k",
                    mew=0.9, zorder=6, clip_on=False)
            shown.append(f"{e} {v:+.1f}%")
    ax.set_title("satellite: Swiss-mean thresholded mask - NWCSAF mask; "
                 + ("now: " + ",  ".join(shown) if shown else "-"),
                 fontsize=10, loc="left")


def frame_path(t):
    return FRAME_DIR / f"clct_contingency_{t:%Y%m%d%H}.png"


def draw_frame(t):
    slot = match_slot(t)
    obs, S = observed_fraction(slot)
    clct = {e: load_clct(e, t) for e in EXPS}
    common = in_domain & np.isfinite(obs)
    for e in EXPS:
        common &= np.isfinite(clct[e])

    tau_px = tau_field(S)
    obs_cloudy = obs >= 50.0
    mod_cloudy = {e: clct[e] >= tau_px for e in EXPS}
    cat = {e: np.where(common, contingency(mod_cloudy[e], obs_cloudy), np.nan)
           for e in EXPS}
    obs_m = np.where(common, obs_cloudy.astype("float64"), np.nan)
    mod_m = {e: np.where(common, mod_cloudy[e].astype("float64"), np.nan)
             for e in EXPS}

    NCOL, FIG_W = 3, 16.5
    _map_h = (FIG_W / NCOL) * 0.90 * (DOMAIN_EXTENT[3] - DOMAIN_EXTENT[2]) \
        / (DOMAIN_EXTENT[1] - DOMAIN_EXTENT[0])
    ROW_H, TOP_H, BIAS_H = _map_h + 0.62, 0.55, 1.85
    fig = Figure(figsize=(FIG_W, 2 * ROW_H + 2 * BIAS_H + TOP_H))
    gs = fig.add_gridspec(4, NCOL,
                          height_ratios=[ROW_H, ROW_H, BIAS_H, BIAS_H])

    def add(r_, c_, field, cmap, title, norm=None):
        ax = fig.add_subplot(gs[r_, c_], projection=ccrs.PlateCarree())
        if norm is None:
            ax.pcolormesh(LON2, LAT2, field, transform=ccrs.PlateCarree(),
                          cmap=cmap, vmin=0, vmax=1, shading="nearest")
        else:
            ax.pcolormesh(LON2, LAT2, field, transform=ccrs.PlateCarree(),
                          cmap=cmap, norm=norm, shading="nearest")
        mask_outside(ax)
        ax.coastlines("10m", linewidth=0.5, zorder=3)
        ax.add_feature(cfeature.BORDERS, linewidth=0.5, zorder=3)
        ax.set_extent(DOMAIN_EXTENT, crs=ccrs.PlateCarree())
        ax.set_title(title, fontsize=9)
        return ax

    # row 0: the three masks
    add(0, 0, obs_m, MASK_CMAP, f"NWCSAF cloud mask, slot {slot:%H:%M}")
    for k, e in enumerate(EXPS):
        add(0, k + 1, mod_m[e], MASK_CMAP,
            f"{e} CLCT >= tau(condition) -> mask")
    # row 1: one contingency map per experiment
    for k, e in enumerate(EXPS):
        add(1, k, cat[e], CAT_CMAP, f"{e} against NWCSAF", norm=CAT_NORM)

    ax_leg = fig.add_subplot(gs[1, len(EXPS)])
    ax_leg.axis("off")
    ax_leg.legend(handles=[Patch(fc=CAT_COLOURS[i], ec="k", lw=0.4, label=CAT_LABEL[i])
                           for i in (1, 2, 3, 0)]
                  + [Patch(fc=NODATA_COLOUR, ec="k", lw=0.4,
                           label="not observed (parallax gap / snow-ice screen)"),
                     Patch(fc="#5b7fa6", ec="k", lw=0.4, label="cloudy (mask panels)")],
                  loc="center", ncol=1, fontsize=9.5, frameon=False)

    ax1 = fig.add_subplot(gs[2, :])
    draw_thr_bias(ax1, t)
    ax2 = fig.add_subplot(gs[3, :])
    draw_movero_bias(ax2, t)
    ax2.set_xlabel("valid time [UTC]")

    _tau = ", ".join(f"{k} {v:g}%" for k, v in TAU_COND.items())
    fig.suptitle(f"Cloud mask contingency, {t:%Y-%m-%d %H:%M} UTC   "
                 f"(tau by retrieval condition: {_tau}; 3 km, no coarsening)",
                 fontsize=12)
    fig.tight_layout(rect=[0, 0, 1, 0.955], h_pad=1.4)
    fig.savefig(frame_path(t), dpi=FRAME_DPI)
    return {"time": t}


def _job(t):
    try:
        return t, draw_frame(t), None
    except Exception as e:
        return t, None, f"{type(e).__name__}: {e}"


_times = list(SER.index.to_pydatetime())
print(f"rendering {len(_times)} frames on {N_WORKERS} workers -> {FRAME_DIR.resolve()}")
rows, failed = [], []
with ProcessPoolExecutor(max_workers=N_WORKERS) as ex:
    futs = [ex.submit(_job, t) for t in _times]
    for done, fu in enumerate(as_completed(futs), 1):
        t, res, err = fu.result()
        (failed if err else rows).append((t, err) if err else res)
        if err:
            print(f"  FAILED {t:%m-%d %Hh}: {err}", flush=True)
        if done % 24 == 0:
            print(f"  {done}/{len(_times)}", flush=True)
print(f"\n{len(rows)} frames written, {len(failed)} failed")

(FRAME_DIR / "provenance.txt").write_text(
    f"experiments {', '.join(EXPS)}\n"
    f"period      {SER.index[0]} .. {SER.index[-1]} ({len(SER)} hours)\n"
    f"threshold   tau by retrieval condition, per pixel: {TAU_COND}, "
    f"fallback {TAU_FALLBACK}%\n"
    f"            fitted on operational KENDA-CH1 det FG vs NWCSAF,\n"
    f"            Sep 2024 - Sep 2026, 16795 scenes (CLCT_tau_conditions.ipynb)\n"
    f"observation NWCSAF cma, slot T-{int(NWCSAF_SLOT_LAG.total_seconds() // 60)}min,\n"
    f"            parallax-corrected, snow/ice screened\n"
    f"panels      row 1: NWCSAF mask | 801 mask | 802 mask\n"
    f"            row 2: 801 contingency | 802 contingency\n"
    f"            rows 3-4: satellite thresholded bias | MOVERO {VERIF_PARAM} "
    f"{VERIF_SCORE}\n"
    f"grid        native 3 km, no coarsening\n"
    f"notebook    CLCT_contingency_frames.ipynb\n")

fv = Path("frame_viewer.py")
if not fv.exists():
    fv = Path("scripts/frame_viewer.py")
if fv.exists():
    subprocess.run([sys.executable, str(fv), str(FRAME_DIR), "--write-only"], check=False)
    print(f"\nStep through them with:\n  python3 {fv} {FRAME_DIR}")